# Общие функции
Этот служебный ноутбук автоматически подключают остальные ноутбуки.
Здесь нет загрузки данных, обучения, сетевых запросов или инференса при подключении.
Общая предобработка и архитектура гарантируют одинаковое поведение обучения и submission.


## Импорты и корень проекта

In [ ]:
from pathlib import Path
import os, sys, json, random, hashlib, time, platform, subprocess, math, io
from contextlib import nullcontext
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance, ImageFilter
from scipy.special import expit, logit
from scipy.optimize import minimize_scalar
from tqdm.auto import tqdm
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights


def find_project_root(start=None):
    from pathlib import Path
    import os

    current = Path(start or Path.cwd()).resolve()
    explicit = os.environ.get("AVITO_PROJECT_ROOT")
    candidates = (
        [Path(explicit).expanduser().resolve()] if explicit else [current, *current.parents]
    )
    for candidate in candidates:
        marker = (candidate / "avito_project.json").is_file()
        layout = (candidate / "notebooks" / "_shared.ipynb").is_file() and (
            candidate / "requirements.txt"
        ).is_file()
        if marker or layout:
            return candidate
    raise RuntimeError(
        f"Не найден корень Avito_test. Текущая папка: {current}. "
        "Перенеси notebooks/ целиком, requirements.txt и avito_project.json в папку проекта. "
        "На VM выполни в отдельной ячейке: %cd /home/slomauh/ml/Avito_test . "
        "Для другого расположения задай переменную окружения AVITO_PROJECT_ROOT. "
        + (f"Сейчас AVITO_PROJECT_ROOT={explicit!r}; проверь этот путь." if explicit else "")
    )


ROOT = find_project_root()
SEED = 42
EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}


## Сохранение, хеши и воспроизводимость

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for part in iter(lambda: f.read(1024 * 1024), b""):
            h.update(part)
    return h.hexdigest()


def save_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    tmp.replace(path)


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


def environment():
    import importlib.metadata

    names = [
        "torch",
        "torchvision",
        "numpy",
        "pandas",
        "pillow",
        "opencv-python-headless",
        "ijson",
        "scipy",
    ]
    versions = {}
    for name in names:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            pass
    return {
        "python": sys.version,
        "platform": platform.platform(),
        "packages": versions,
        "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }


## Изображения и предобработка

In [ ]:
def open_rgb(path):
    # Ориентация определяется сохранёнными пикселями: EXIF-поворот не применяем.
    with Image.open(path) as im:
        if im.mode in ("RGBA", "LA") or "transparency" in im.info:
            rgba = im.convert("RGBA")
            bg = Image.new("RGBA", rgba.size, "white")
            im = Image.alpha_composite(bg, rgba)
        return im.convert("RGB")


def script_of(text):
    cyr = any("\u0400" <= ch <= "\u052f" for ch in text)
    lat = any(("a" <= ch.lower() <= "z") for ch in text)
    if cyr and lat:
        return "mixed"
    if cyr:
        return "cyrillic"
    if lat:
        return "latin"
    if any(ch.isdigit() for ch in text) and not any(ch.isalpha() for ch in text):
        return "numeric"
    return "other"


def letterbox(im, height=48, width=192):
    # Сохраняем пропорции. Симметричные поля не создают подсказку классу.
    scale = min(width / im.width, height / im.height)
    size = (
        max(1, min(width, round(im.width * scale))),
        max(1, min(height, round(im.height * scale))),
    )
    im = im.resize(size, Image.Resampling.BILINEAR)
    canvas = Image.new("RGB", (width, height), (127, 127, 127))
    canvas.paste(im, ((width - size[0]) // 2, (height - size[1]) // 2))
    arr = np.asarray(canvas, dtype=np.float32) / 255.0
    arr = (arr - np.array([0.485, 0.456, 0.406], np.float32)) / np.array(
        [0.229, 0.224, 0.225], np.float32
    )
    return torch.from_numpy(arr.transpose(2, 0, 1).copy())


## Аугментации train

In [ ]:
def small_rotation(im, degrees=0.0, probability=0.5):
    if not 0 <= degrees <= 15 or not 0 <= probability <= 1:
        raise ValueError("Небольшой наклон: degrees в [0,15], probability в [0,1]")
    if degrees == 0 or probability == 0 or random.random() >= probability:
        return im
    # Расширяем холст, чтобы не обрезать буквы. Цвет полей оцениваем по краям.
    arr = np.asarray(im.convert("RGB"))
    edges = np.concatenate([arr[0], arr[-1], arr[:, 0], arr[:, -1]], axis=0)
    fill = tuple(int(v) for v in np.median(edges, axis=0))
    return im.rotate(
        random.uniform(-degrees, degrees),
        resample=Image.Resampling.BICUBIC,
        expand=True,
        fillcolor=fill,
    )


def augment(im, rotation_degrees=0.0, rotation_probability=0.5):
    im = small_rotation(im, rotation_degrees, rotation_probability)
    if random.random() < 0.4:
        im = ImageEnhance.Contrast(im).enhance(random.uniform(0.5, 1.5))
    if random.random() < 0.4:
        im = ImageEnhance.Brightness(im).enhance(random.uniform(0.6, 1.4))
    if random.random() < 0.25:
        im = im.filter(ImageFilter.GaussianBlur(random.uniform(0.1, 1.1)))
    if random.random() < 0.25:
        scale = random.uniform(0.4, 0.9)
        im = im.resize(
            (max(2, round(im.width * scale)), max(2, round(im.height * scale))),
            Image.Resampling.BILINEAR,
        )
    if random.random() < 0.25:
        stream = io.BytesIO()
        im.save(stream, format="JPEG", quality=random.randint(35, 90))
        stream.seek(0)
        with Image.open(stream) as jpeg:
            im = jpeg.convert("RGB")
    return im


## Manifest и загрузчик данных

In [ ]:
def load_manifest(path):
    df = pd.read_csv(
        path, keep_default_na=False, dtype={"crop_id": str, "source_id": str, "group_id": str}
    )
    required = {
        "crop_id",
        "source_id",
        "group_id",
        "source",
        "split",
        "path",
        "text",
        "script",
        "pixel_hash",
    }
    if not required.issubset(df.columns) or len(df) == 0:
        raise ValueError(f"Пустой или неверный manifest: {path}")
    if df.crop_id.duplicated().any():
        raise ValueError("Повторяются crop_id")
    for key in ["group_id", "pixel_hash"]:
        if (df.groupby(key).split.nunique() > 1).any():
            raise ValueError(f"Утечка между частями выборки по {key}")
    missing = [p for p in df.path if not (ROOT / p).is_file()]
    if missing:
        raise FileNotFoundError(f"Не найдены кропы, например: {missing[:3]}")
    return df


class CropDataset(Dataset):
    def __init__(
        self,
        frame,
        height=48,
        width=192,
        training=False,
        rotation_degrees=0.0,
        rotation_probability=0.5,
    ):
        self.frame = frame.reset_index(drop=True)
        self.height, self.width, self.training = height, width, training
        self.rotation_degrees, self.rotation_probability = rotation_degrees, rotation_probability

    def __len__(self):
        return len(self.frame) if self.training else 2 * len(self.frame)

    def __getitem__(self, idx):
        base_idx = idx if self.training else idx // 2
        row = self.frame.iloc[base_idx]
        im = open_rgb(ROOT / row.path)
        if self.training:
            im = augment(im, self.rotation_degrees, self.rotation_probability)
        # Метка задаётся после аугментаций: они не должны определять класс.
        y = random.randrange(2) if self.training else idx % 2
        if y:
            im = im.transpose(Image.Transpose.ROTATE_180)
        x = letterbox(im, self.height, self.width)
        return x, float(y), str(row.crop_id) + f"__r{180*y}"


def make_loader(frame, cfg, training=False):
    # Dataset определён в ноутбуке: workers=0 совместим и с Windows, и с Linux.
    # Это стартовая надёжная настройка; ускорять загрузчик будем по измерениям.
    ds = CropDataset(
        frame,
        cfg["height"],
        cfg["width"],
        training,
        cfg.get("rotation_degrees", 0.0),
        cfg.get("rotation_probability", 0.5),
    )
    return DataLoader(
        ds,
        batch_size=cfg["batch_size"],
        shuffle=training,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
    )


## Архитектура и предсказания

In [ ]:
def build_model(pretrained=False):
    weights = MobileNet_V3_Small_Weights.IMAGENET1K_V1 if pretrained else None
    model = mobilenet_v3_small(weights=weights)
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 1)
    return model


@torch.inference_mode()
def predict_loader(model, loader, device, amp=False, rotated=False):
    model.eval()
    parts = []
    for x, y, ids in tqdm(loader, desc="Предсказание", leave=False):
        x = x.to(device, non_blocking=True)
        with torch.autocast(
            device_type="cuda", dtype=torch.float16, enabled=amp and device.type == "cuda"
        ):
            z = model(x).flatten().float()
            zr = model(torch.flip(x, dims=(-2, -1))).flatten().float() if rotated else None
        part = pd.DataFrame({"sample_id": list(ids), "y": y.numpy(), "logit": z.cpu().numpy()})
        if rotated:
            part["logit_rot"] = zr.cpu().numpy()
        parts.append(part)
    if not parts:
        raise ValueError("Пустая выборка для предсказания")
    return pd.concat(parts, ignore_index=True)


## Вероятности, метрики и калибровка

In [ ]:
def probabilities(pred, temperature=1.0, tta=False):
    if not np.isfinite(temperature) or temperature <= 0:
        raise ValueError("Температура должна быть положительной")
    p = expit(np.asarray(pred["logit"], dtype=np.float64) / temperature)
    if tta:
        # Для повернутого входа вероятность класса 180 переводим в систему исходного.
        p = (p + 1 - expit(np.asarray(pred["logit_rot"], dtype=np.float64) / temperature)) / 2
    return p


def metrics(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    if (
        y.shape != p.shape
        or y.size == 0
        or not np.isfinite(p).all()
        or not np.isin(y, [0, 1]).all()
        or not ((p >= 0) & (p <= 1)).all()
    ):
        raise ValueError("Неверные метки или вероятности")
    brier = float(np.mean((p - y) ** 2))
    return {
        "brier": brier,
        "score": 1 - brier,
        "accuracy": float(np.mean((p >= 0.5) == y)),
        "n": int(y.size),
    }


def fit_temperature(pred, tta=False):
    def objective(log_t):
        return metrics(pred.y, probabilities(pred, np.exp(log_t), tta))["brier"]

    result = minimize_scalar(objective, bounds=(np.log(0.1), np.log(10.0)), method="bounded")
    candidates = [1.0, float(np.exp(result.x)), 0.1, 10.0]
    return min(candidates, key=lambda t: objective(np.log(t)))


def bootstrap_score_ci(pred, p, repetitions=500, seed=SEED):
    # Кропы одной фотографии зависимы: bootstrap по исходным изображениям.
    temp = pred[["group_id", "y"]].copy()
    temp["loss"] = (np.asarray(p) - temp.y) ** 2
    groups = temp.groupby("group_id").loss.agg(["sum", "count"]).to_numpy()
    rng = np.random.default_rng(seed)
    scores = []
    for _ in range(repetitions):
        sample = groups[rng.integers(0, len(groups), size=len(groups))]
        scores.append(1 - sample[:, 0].sum() / sample[:, 1].sum())
    return np.quantile(scores, [0.025, 0.975]).tolist()


def attach_metadata(pred, manifest):
    result = pred.copy()
    result["crop_id"] = result.sample_id.str.rsplit("__r", n=1).str[0]
    result = result.merge(
        manifest[["crop_id", "group_id", "script", "text", "source", "path"]],
        on="crop_id",
        how="left",
        validate="many_to_one",
    )
    if result.group_id.isna().any():
        raise ValueError("Не удалось сопоставить предсказания с manifest")
    return result


## Проверка тестовых файлов и CSV

In [ ]:
def test_files(test_dir, sample_path, expected_count=20000):
    test_dir = Path(test_dir).expanduser().resolve()
    if not test_dir.is_dir():
        raise FileNotFoundError(f"Папка теста не существует: {test_dir}. Исправь TEST_DIR.")
    sample = pd.read_csv(sample_path, dtype={"image_id": str}, keep_default_na=False)
    if list(sample.columns) != ["image_id", "p_180"]:
        raise ValueError("sample_submission должен содержать image_id,p_180")
    ids = sample.image_id.tolist()
    if len(ids) != expected_count or len(set(ids)) != len(ids) or any(not x for x in ids):
        raise ValueError(f"Нужно {expected_count} уникальных image_id")
    by_name, by_stem = {}, {}
    for path in sorted(test_dir.rglob("*")):
        if path.is_file() and path.suffix.lower() in EXTENSIONS:
            if path.name in by_name:
                raise ValueError(f"Одинаковые имена файлов в подпапках: {path.name}")
            by_name[path.name] = path
            by_stem.setdefault(path.stem, []).append(path)
    if not by_name:
        raise ValueError(
            f"В {test_dir} не найдено изображений. Проверь TEST_DIR и распакован ли архив теста Авито. "
            f"Поддерживаемые расширения: {sorted(EXTENSIONS)}. "
            "Нужны готовые кропы Авито, а не train/validation HierText."
        )
    matched, missing, used = [], [], set()
    for image_id in ids:
        if image_id in by_name:
            chosen = by_name[image_id]
        else:
            candidates = by_stem.get(image_id, [])
            if len(candidates) > 1:
                raise ValueError(f"Неоднозначный ID {image_id}: {[str(p) for p in candidates]}")
            if not candidates:
                missing.append(image_id)
                continue
            chosen = candidates[0]
        if chosen in used:
            raise ValueError(f"Разные image_id ссылаются на один файл: {chosen}")
        used.add(chosen)
        matched.append(chosen)
    extra = sorted(str(p.relative_to(test_dir)) for p in set(by_name.values()) - used)
    if missing or extra:
        raise ValueError(
            f"Не совпадают файлы и sample в {test_dir}: найдено изображений {len(by_name)}, "
            f"ожидается ID {len(ids)}; missing={missing[:5]} (всего {len(missing)}), "
            f"extra={extra[:5]} (всего {len(extra)}). "
            "Поддерживаются ID как с расширением, так и без него."
        )
    # Возвращаем неизменённые ID и порядок шаблона; расширение не добавляем в CSV.
    return ids, matched


def write_submission(ids, p, path, expected_count=20000):
    p = np.asarray(p, float)
    if p.shape != (len(ids),) or len(ids) != expected_count or len(set(ids)) != len(ids):
        raise ValueError("Неверная длина или повторяющиеся image_id")
    if not np.isfinite(p).all() or not ((p >= 0) & (p <= 1)).all():
        raise ValueError("p_180 вне [0,1] или содержит NaN/Inf")
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        raise FileExistsError(f"Файл уже существует: {path}; выбери новое имя")
    pd.DataFrame({"image_id": ids, "p_180": p}).to_csv(path, index=False, float_format="%.10f")
    check = pd.read_csv(path, dtype={"image_id": str}, keep_default_na=False)
    assert list(check.columns) == ["image_id", "p_180"] and check.image_id.tolist() == ids
    return path


## Предсказания для файлов

In [ ]:
@torch.inference_mode()
def predict_images(model, paths, cfg, deployment, device, batch_size=128):
    model.eval()
    output = []
    for start in tqdm(range(0, len(paths), batch_size), desc="Submission"):
        # Поворот для TTA выполняется после той же предобработки, что при калибровке.
        x = torch.stack(
            [
                letterbox(open_rgb(p), cfg["height"], cfg["width"])
                for p in paths[start : start + batch_size]
            ]
        ).to(device)
        pred = {"logit": model(x).flatten().float().cpu().numpy()}
        if deployment["tta"]:
            pred["logit_rot"] = model(torch.flip(x, dims=(-2, -1))).flatten().float().cpu().numpy()
        output.extend(probabilities(pred, deployment["temperature"], deployment["tta"]))
    return np.asarray(output)


## Идентичность начальных весов

In [ ]:
def weights_digest(model):
    h = hashlib.sha256()
    for key, value in sorted(model.state_dict().items()):
        h.update(key.encode())
        h.update(value.detach().cpu().numpy().tobytes())
    return h.hexdigest()
